# 00 — Viết qrels (bảng chân lý cho retrieval)

**qrels = bảng chân lý**: với mỗi câu hỏi, bài nào trả lời được, bài nào chỉ liên quan, bài nào không.

Không có qrels thì mọi metric (MRR, nDCG@k) chỉ là con số vô nghĩa. Đây là file bạn sẽ quay lại
nhiều lần: viết 12 query trước, sau đó mở rộng lên 40-50.

## Thang điểm (áp dụng NHẤT QUÁN cho mọi query)

| Điểm | Nghĩa | Cách tự hỏi |
|---|---|---|
| **2** | Bài **trả lời trực tiếp** câu hỏi | "Đọc xong bài này, người hỏi có biết đáp án không?" |
| **1** | **Liên quan một phần** — cùng chủ đề nhưng không trả lời trực tiếp | "Bài này nói về chuyện khác nhưng có nhắc tới khái niệm trong câu hỏi" |
| **0** | Không liên quan (chỉ chấm khi bạn đã thực sự xem bài đó) | "Bài này chẳng giúp gì cho câu hỏi" |

## Quy trình 3 bước cho MỖI câu hỏi

1. `search("...")` — dùng **câu hỏi đầy đủ cũng được**, công cụ tự tách từ
2. `show("<doc_id>")` — đọc toàn văn bài còn phân vân
3. `judge(...)` — chấm điểm

Cuối cùng: `save()` → ghi ra `eval/qrels.csv`.

In [1]:
# ── Khởi tạo: nạp corpus + chỉ mục (chạy 1 lần, ~5-10 giây) ────
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from explore import load_docs, build_index, search as _search   # dùng CHUNG code với CLI

QRELS = ROOT / "eval" / "qrels.csv"
_docs, SOURCE = load_docs()
INDEX = build_index(_docs)
docs = pd.DataFrame(_docs)
print(f"Corpus: {len(docs)} bài · {docs['chars'].sum()/1e6:.2f} MB · nguồn: {SOURCE}")

Corpus: 179 bài · 2.47 MB · nguồn: curated (data\processed\curated.jsonl)


In [2]:
# ── Công cụ chấm điểm (không cần sửa) ──────────────────────────
def search(query: str, k: int = 8):
    """Xếp hạng bài theo truy vấn — dùng câu hỏi đầy đủ hoặc vài từ khoá đều được."""
    rows = _search(query, _docs, k=k, index=INDEX)
    if not rows:
        return f"Không tìm thấy bài nào cho '{query}'"
    return pd.DataFrame(rows)[["doc_id", "title", "chars", "matched", "score", "snippet"]]

def show(doc_id, limit: int = 2000):
    """In toàn văn (hoặc `limit` ký tự đầu) một bài để chấm điểm."""
    row = docs[docs["doc_id"] == str(doc_id)]
    if row.empty:
        print(f"Không có doc_id {doc_id}"); return
    r = row.iloc[0]
    print(f"# {r['title']}\n{r['url']}\n({r['chars']:,} ký tự)\n")
    print(r["text"][:limit] + ("\n\n[... còn nữa — tăng limit để đọc thêm]" if r["chars"] > limit else ""))

# sổ chấm điểm (nạp lại nếu đã chấm từ trước)
if QRELS.exists():
    jud = pd.read_csv(QRELS, dtype={"doc_id": str}).dropna(subset=["relevance"]).to_dict("records")
    print(f"Đã nạp {len(jud)} dòng có sẵn từ {QRELS.relative_to(ROOT)}")
else:
    jud = []
    print("Chưa có file qrels — sẽ tạo khi save()")

def judge(query_id: str, query: str, grades: dict, query_class: str):
    """Chấm điểm cho MỘT câu hỏi. grades = {"doc_id": điểm, ...}

    ví dụ: judge("q001", "Học có giám sát là gì?", {"24329": 2, "19895408": 1}, "literal")
    """
    assert query_class in {"literal", "paraphrase", "multi-doc"}, "query_class: literal / paraphrase / multi-doc"
    for did, rel in grades.items():
        assert str(did) in set(docs["doc_id"]), f"doc_id {did} không tồn tại trong corpus!"
        assert rel in (0, 1, 2), "relevance chỉ nhận 0 / 1 / 2"
        jud.append({"query_id": query_id, "query": query, "doc_id": str(did),
                    "relevance": rel, "query_class": query_class})
    print(f"✓ {query_id}: chấm {len(grades)} bài ({query_class}) → tổng {len(jud)} dòng")

def progress():
    if not jud:
        print("Chưa chấm query nào."); return
    df = pd.DataFrame(jud)
    print(df.groupby(["query_id", "query_class"]).agg(số_bài=("doc_id", "count")).reset_index().to_string(index=False))
    print("\nTheo lớp:", df.groupby("query_class")["query_id"].nunique().to_dict())
    print(f"Tổng: {df['query_id'].nunique()} query · {len(df)} dòng")

def save():
    """Ghi ra eval/qrels.csv."""
    if not jud:
        print("Chưa chấm query nào — không ghi đè file qrels."); return
    df = pd.DataFrame(jud)[["query_id", "query", "doc_id", "relevance", "query_class"]]
    QRELS.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(QRELS, index=False)
    print(f"Đã lưu {len(df)} dòng → {QRELS.relative_to(ROOT)}")
print("Sẵn sàng: search() · show() · judge() · progress() · save()")

Đã nạp 0 dòng có sẵn từ eval\qrels.csv
Sẵn sàng: search() · show() · judge() · progress() · save()


## Bước 1 — Tìm bài ứng viên

Gõ **câu hỏi đầy đủ** cũng được (công cụ tự tách từ + bỏ từ dừng), hoặc gõ vài từ khoá.

> ⚠️ Công cụ này chỉ để **tìm ứng viên**, không phải để chấm điểm. Nó cố tình đơn giản
> (IDF + chuẩn hoá độ dài) — phiên bản BM25 bạn sắp code sẽ tốt hơn. Nếu kết quả xếp hạng
> chưa chuẩn, đó chính là dữ liệu quý cho phần so sánh sau này.

In [3]:
search("Học có giám sát là gì?")

,doc_id,title,chars,matched,score,snippet
0,19895408,Học tự giám sát,8010,3,210.61,Học tự giám sát (self-supervised learning hay ...
1,19893713,Học đặc trưng,24688,3,170.05,"Trong học máy, học đặc trưng (feature learning..."
2,3339820,Học sâu,57004,3,137.83,"Học sâu (tiếng Anh: deep learning, còn gọi là ..."
3,24345,Học không có giám sát,1866,3,129.94,Học không có giám sát (tiếng Anh: unsupervised...
4,20014464,Thuật ngữ trí tuệ nhân tạo,42756,3,119.03,"... đến ngành trí tuệ nhân tạo (AI), các phân ..."
5,19950271,Học vấn về AI,8233,3,107.72,Học vấn về AI hay học vấn về trí tuệ nhân tạo ...
6,24278,Học máy,19947,3,97.68,Học máy (tiếng Anh: Machine learning) là một l...
7,20012838,Tốc độ học,7071,3,91.16,"Tốc độ học (tiếng Anh: learning rate, còn được..."


## Bước 2 — Đọc bài còn phân vân

Tiêu đề đúng chủ đề → gần như chắc chắn điểm 2. Bài chỉ *nhắc tới* → phải đọc mới biết.

In [4]:
show("19895408", limit=1200)

# Học tự giám sát
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_t%E1%BB%B1_gi%C3%A1m_s%C3%A1t
(8,010 ký tự)

Học tự giám sát (self-supervised learning hay SSL) là một kỹ thuật trong học máy mà trong đó, một mô hình được huấn luyện trên một tác vụ bằng cách sử dụng chính dữ liệu để tạo ra các tín hiệu giám sát, thay vì dựa vào các nhãn bên ngoài do con người cung cấp. Trong bối cảnh của mạng thần kinh nhân tạo, học tự giám sát hướng tới việc khai thác các cấu trúc hoặc mối quan hệ nội tại trong dữ liệu đầu vào để tạo ra các tín hiệu huấn luyện có ý nghĩa. Các nhiệm vụ trong SSL được thiết kế sao cho việc giải quyết yêu cầu nắm bắt được các đặc điểm hoặc mối quan hệ quan trọng trong dữ liệu. Dữ liệu đầu vào thường được tăng cường hoặc biến đổi để tạo ra các cặp mẫu liên quan. Một mẫu sẽ đóng vai trò làm đầu vào, và mẫu còn lại được dùng để hình thành tín hiệu giám sát. Việc tăng cường này có thể bao gồm thêm nhiễu, cắt, xoay, hoặc các biến đổi khác. Học tự giám sát mô phỏng cách con người họ

## Bước 3 — Chấm điểm

Mỗi câu hỏi = 1 lệnh `judge()`, truyền dict `{doc_id: điểm}`.

Ví dụ dưới đây là **câu hỏi mẫu** — sửa theo phán đoán của CHÍNH BẠN rồi bỏ dấu `#`.

In [5]:
# judge("q001", "Học có giám sát là gì?", {
#     "24329":    2,   # bài đúng chủ đề, định nghĩa trực tiếp
#     "19895408": 1,   # "Học tự giám sát" — chủ đề khác, chỉ nhắc để so sánh
#     "20014464": 1,   # "Thuật ngữ AI" — có 1 dòng định nghĩa, không giải thích sâu
# }, "literal")

progress()

Chưa chấm query nào.


## Checklist 12 query đợt 1

- [ ] 4 × **literal** — dùng đúng từ khoá trong bài (*"Mạng nơ-ron tích chập là gì?"*)
- [ ] 4 × **paraphrase** — KHÔNG lặp từ khoá của bài (*"Máy tính học từ ví dụ có nhãn bằng cách nào?"* thay vì *"học có giám sát"*)
- [ ] 4 × **multi-doc** — cần ≥2 bài (*"So sánh học có giám sát và học tự giám sát"* → gold phải có cả 2 bài)

**Chọn chủ đề bạn đã học** → chấm relevance tự tin hơn nhiều.
Xem 179 tiêu đề: `python scripts/explore.py --titles` (chạy ở terminal).

In [6]:
# ── Lưu lại ────────────────────────────────────────────────────
progress()
save()

Chưa chấm query nào.
Chưa chấm query nào — không ghi đè file qrels.
